# Integrating and Running PCN, GPIPD, and CAPQL in the CommonPower Framework

This notebook provides a comprehensive guide to understanding, running, and evaluating the custom reinforcement learning algorithms **PCN**, **GPIPD**, and **CAPQL** integrated into the `CommonPower` framework. This tutorial covers the entire workflow, from setting up the environment to training the models and deploying them for evaluation.

## 1. Setup

This section details the necessary steps to create the correct environment for running the experiments. This includes installing the required Python packages and configuring the Gurobi solver. These steps must be completed before running the rest of the notebook.

### 1.1. Install Dependencies from GitLab Repository

Our project uses a modified version of `commonpower` where we have integrated advanced multi-objective reinforcement learning (MORL) algorithms from the **`morl-baselines`** library. 

The following commands will clone the correct branch `finetuning` and install it in editable mode, therefore any changes to the source code will be directly be reflected in the installed package.

In [ ]:
# 1. Clone the repository from the GitLab
!git clone https://gitlab.lrz.de/cps/cps-power/students/ss-25-group-2.git

# 2. Navigate into the cloned directory
%cd commonpower

# 3. Switch to the correct branch
!git checkout finetuning

# 4. Install the package in editable mode
!pip install -e .

# 5. Navigate back to the root directory
%cd ..

### 1.2. Gurobi Solver Setup

The `CommonPower` framework uses Gurobi as the default solver for optimization problems. You will need to obtain an academic license and install it on your system.

1.  **Get a License**: Obtain a free academic license from the [Gurobi Academic Program](https://www.gurobi.com/academia/academic-program-and-licenses/).
2.  **Install Gurobi**: Follow the [Gurobi Quickstart Guide](https://www.gurobi.com/documentation/quickstart.html) for your operating system to install the solver.
3.  **Activate License**: Retrieve and activate your license as described in the quickstart guide.

### 1.4. Helper Code (from Project Files)

To make this notebook self-contained, the core logic from the project's Python files (`utils.py`, `scenarios.py`, `training.py`, `deployment.py`) is included below. This code defines the experimental setup, including scenarios, algorithms, and training/deployment procedures. Note that it can take some time to run.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum, auto
from abc import ABCMeta, abstractmethod
from pathlib import Path
from typing import List, Optional
from datetime import datetime, timedelta

# CommonPower Imports
from commonpower.core import System, ModelHistory
from commonpower.models.components import *
from commonpower.models.buses import *
from commonpower.models.powerflow import *
from commonpower.data_forecasting import *
from commonpower.control.runners import DeploymentRunner, SingleAgentTrainer
from commonpower.control.controllers import *
from commonpower.control.wrappers import WrapperStack, SingleAgentWrapper
from commonpower.control.safety_layer.penalties import *
from commonpower.control.safety_layer.safety_layers import *
from commonpower.modeling.param_initialization import *
from commonpower.control.logging_utils.loggers import TensorboardLogger

# Corrected imports for policy classes
from commonpower.control.policies.ppo_policy import PPOPolicy
from commonpower.control.policies.sac_policy import SACPolicy
from commonpower.control.policies.pcn_policy import PCNPolicy
from commonpower.control.policies.capql_policy import CAPQLPolicy
from commonpower.control.policies.gpipd_policy import GPIPDPolicy

from commonpower.control.configs.algorithms import AlgorithmBaseConfig, MetaConfig, PPO_Config, SAC_Config, PCN_Config, CAPQL_Config, GPIPD_Config

# turn W&B completely off for this notebook 
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

# CAPQL/Torch load compat (stops the weights_only kwarg crash on deploy)
import torch as th
__orig_torch_load = th.load
def _torch_load_compat(*args, **kwargs):
    kwargs.pop("weights_only", None)
    return __orig_torch_load(*args, **kwargs)
th.load = _torch_load_compat

# --- Contents from utils.py ---
class CEnum(Enum):
    def __str__(self):
        return self.name

class Stage(CEnum):
    Train = auto()
    Deploy = auto()

class Approach(CEnum):
    WithProjectionSafeguard = auto()
    WithReplacementSafeguard = auto()
    OptimalController = auto()

class Penalty(CEnum):
    NoPenalty = auto()
    ConstantPenalty = auto()
    DDPenalty = auto()
    BothPenalties = auto()

class RLAlgorithm(CEnum):
    PPO = auto()
    SAC = auto()
    PCN = auto()
    CAPQL = auto()
    GPIPD = auto()

    def to_policy_class(self):
        return {
            RLAlgorithm.PPO: PPOPolicy,
            RLAlgorithm.SAC: SACPolicy,
            RLAlgorithm.PCN: PCNPolicy,
            RLAlgorithm.CAPQL: CAPQLPolicy,
            RLAlgorithm.GPIPD: GPIPDPolicy
        }.get(self)

# --- Contents from scenarios.py ---
class BaseScenario(metaclass=ABCMeta):
    def __init__(self, mode: Stage, forecaster: Forecaster, data_path: Path, date_format: str):
        super().__init__()
        self.mode = mode
        self.data_path = data_path.resolve()
        self.date_format = date_format
        self.forecaster = forecaster
        self.forecast_frequency = forecaster.frequency
        self.forecast_horizon = forecaster.horizon
        self.sys = self._create_system()

    def get_system(self) -> System:
        return self.sys

    @abstractmethod
    def _create_system(self) -> System:
        pass

class BuildingManagementSystemWithEVScenario(BaseScenario):
    def __init__(self, mode: Stage, forecaster: Forecaster, data_path: Path, date_format: str, use_heat_pump: bool = False, price_buying: float = 0.37, price_selling: float = 0.08):
        self.use_heat_pump = use_heat_pump
        self.price_buying = price_buying
        self.price_selling = price_selling
        super().__init__(mode=mode, forecaster=forecaster, data_path=data_path, date_format=date_format)

    def _create_system(self):
        self.define_data_sources()
        self.load_p_dp = DataProvider(self.p_load_ds, self.forecaster)
        self.load_q_dp = DataProvider(self.q_load_ds, self.forecaster)
        self.price_dp = DataProvider(self.buying_price_ds, self.forecaster)
        self.selling_price_dp = DataProvider(self.selling_price_ds, self.forecaster)
        self.pv_dp = DataProvider(self.pv_ds, self.forecaster)
        n1 = RTPricedBus("MultiFamilyHouse", {'p': (-50, 50), 'q': (-50, 50), 'v': (0.95, 1.05), 'd': (-15, 15)})
        n1.add_data_provider(self.price_dp).add_data_provider(self.selling_price_dp)
        m1 = ExternalGrid("ExternalGrid")
        r1 = RenewableGen("PV1").add_data_provider(self.pv_dp)
        d1 = Load("Load1").add_data_provider(self.load_p_dp).add_data_provider(self.load_q_dp)
        capacity = 5
        ess_initializer = ConstantInitializer(0.5 * capacity)
        e1 = ESSLinear("ESS1", {'p': (-1.5, 1.5), 'q': (0, 0), 'soc': (0.1 * capacity, 0.9 * capacity), "soc_init": ess_initializer})
        n1.add_node(d1).add_node(r1).add_node(e1)
        return System(power_flow_model=PowerBalanceModel()).add_node(n1).add_node(m1)

    def define_data_sources(self):
        self.p_load_ds = CSVDataSource(self.data_path / 'ICLR_load_with_ev.csv', datetime_format=self.date_format, resample=self.forecast_frequency)
        self.q_load_ds = ConstantDataSource({"q": 0.0}, date_range=self.p_load_ds.get_date_range(), frequency=self.forecast_frequency)
        self.buying_price_ds = CSVDataSource(self.data_path / 'ToU_prices.csv', datetime_format=self.date_format, resample=self.forecast_frequency)
        self.selling_price_ds = ConstantDataSource({"psis": self.price_selling}, date_range=self.buying_price_ds.get_date_range(), frequency=self.forecast_frequency)
        self.pv_ds = CSVDataSource(self.data_path / 'ICLR_pv.csv', datetime_format=self.date_format, resample=self.forecast_frequency).apply_to_column("p", lambda x: -x)

class Scenario(CEnum):
    AddedEVScenario = BuildingManagementSystemWithEVScenario

def create_scenario_and_controller(stage: Stage, scenario_constructor: BaseScenario, approach: Approach, penalty: Penalty, forecast_length: int, forecaster: Forecaster):
    current_path = Path('./commonpower/finetuning/data')
    date_format = "%Y-%m-%d %H:%M:%S"
    if approach == Approach.WithProjectionSafeguard:
        if penalty == Penalty.DDPenalty:
            safety_penalty = DistanceDependingPenalty(penalty_factor=1.0)
        else:
            safety_penalty = NoPenalty()
        safeguard = ActionProjectionSafetyLayer(penalty=safety_penalty)
    else:
        safeguard = None
    
    train_scenario = scenario_constructor(mode=stage, data_path=current_path, date_format=date_format, forecaster=forecaster)
    sys = train_scenario.get_system()

    if approach == Approach.OptimalController:
        controller = OptimalController(name="agent1")
    else:
        controller = RLController(name="agent1", safety_layer=safeguard, obs_handler=ObservationHandler(num_forecasts=forecast_length))
    
    controller.add_entity(sys.nodes[0])
    return sys

# --- Contents from training.py ---
def run_training(run_id: str, algo_config: AlgorithmBaseConfig, forecast_horizon: timedelta, episode_length: int, train_sys: System, rl_algorithm: RLAlgorithm, seed: int, n_episodes: int, start_time: str, scalarisation_fn: Optional[callable], ref_point: Optional[np.ndarray]):
    total_steps = n_episodes * episode_length
    train_config = MetaConfig(total_steps=total_steps, seed=seed, policy_class=rl_algorithm.to_policy_class(), algorithm_config=algo_config)
    tb_log_dir = os.path.join(os.getcwd(), 'tensorboard', run_id, str(seed))
    logger = TensorboardLogger(log_dir=str(tb_log_dir))
    
    # Set logger attribute manually for this notebook
    setattr(logger, "project_name", "tutorial")
    setattr(logger, "entity_name",  "local")
    setattr(logger, "run_name",     f"{run_id}-seed{seed}")
    setattr(logger, "group",        run_id)
    
    model_dir = os.path.join(os.getcwd(), 'models', run_id, str(seed))
    if not os.path.exists(model_dir):
        os.makedirs(model_dir, exist_ok=True)
    
    wrappers = WrapperStack().add(SingleAgentWrapper)
    runner = SingleAgentTrainer(sys=train_sys, wrapper=wrappers.get_stack(), alg_config=train_config, horizon=forecast_horizon, episode_length=episode_length, logger=logger, save_path=model_dir, seed=seed, scalarisation_fn=scalarisation_fn, ref_point=ref_point)
    print(f"--- Starting Training for {run_id}, Seed {seed} ---")
    runner.run(fixed_start=datetime.strptime(start_time, "%Y-%m-%d %H:%M:%S"))
    print(f"--- Finished Training for {run_id}, Seed {seed} ---")

# --- Contents from deployment.py ---

# add placeholders for PCN manually in this notebook
setattr(DeploymentRunner, "desired_return", None)
setattr(DeploymentRunner, "desired_horizon", None)

def run_deployment(scenario: System, train_seed: int, horizon: timedelta, approach: Approach,
                   rl_algorithm: RLAlgorithm, algo_config: AlgorithmBaseConfig, save_path: str,
                   eval_period: str, n_eval_steps: int, eval_seed: int):
    alg_config = None
    if approach is not Approach.OptimalController:
        alg_config = MetaConfig(total_steps=1, seed=train_seed,
                                policy_class=rl_algorithm.to_policy_class(), algorithm_config=algo_config)
    
    model_dir = Path.cwd() / "models" / save_path / str(train_seed)
    results_dir = Path.cwd() / "results" / save_path / str(train_seed)
    results_dir.mkdir(parents=True, exist_ok=True)

    wrappers = WrapperStack()
    if approach is not Approach.OptimalController:
        wrappers.add(SingleAgentWrapper)
        system_nodes = getattr(scenario, "nodes")
        rl_controller = getattr(system_nodes[0], "controller")
        setattr(rl_controller, "load_path", str(model_dir))

    history = ModelHistory([scenario])
    deployer = DeploymentRunner(sys=scenario, horizon=horizon, history=history, seed=eval_seed,
                                alg_config=alg_config, wrapper=wrappers.get_stack(), continuous_control=True)

    if rl_algorithm == RLAlgorithm.PCN:
        # pull from globals defined in deployment cell if present; otherwise leave as None
        desired_ret = globals().get("pcn_desired_return", None)
        desired_hor = globals().get("pcn_desired_horizon", None)
        if desired_ret is not None:
            deployer.desired_return  = desired_ret
        if desired_hor is not None:
            deployer.desired_horizon = desired_hor

    deployer.set_start_time(datetime.strptime(eval_period, "%Y-%m-%d %H:%M:%S"))
    print(f"--- Starting Deployment for {save_path}, Seed {train_seed} ---")
    deployer.run(n_steps=n_eval_steps)
    print(f"--- Finished Deployment for {save_path}, Seed {train_seed} ---")

    # Keep full log in memory for plotting; still save a CSV for backup
    log_df = pd.DataFrame(deployer.deployment_log)
    results_dir.mkdir(parents=True, exist_ok=True)
    log_df.to_csv(results_dir / "seed_results.csv", index=False)

    return history, log_df

## 2. Understanding Multi-Objective RL Parameters (for PCN, GPIPD)

Multi-objective reinforcement learning (MORL) algorithms like PCN and GPIPD require a **reference point** to evaluate the quality of a policy. This point helps normalize the objectives and is used in metrics like hypervolume calculation. In our setup, this is handled by the `ref_point` argument in the training script. This reference point is derived from two key calculations:

1.  **Worst-Case Cost**: The maximum possible daily cost in a scenario. This serves as the lower bound for the cost-related reward objective (since reward = -cost).
2.  **Action Space Diameter**: The maximum possible change in action, representing the range of control.

Below is a summary of how these values are calculated for the `AddedEVScenario`.

### 2.1. Worst-Case Cost Calculation

To find the upper bound on cost, we simulate a worst-case scenario where:
- We **pay** for all imported power (positive net load).
- We **receive no revenue** for exported power (negative net load is treated as zero).
- The system has no battery storage to optimize energy usage.

The net load is calculated as: `Net Load = (Building Load + EV Load) - PV Generation`.

The cost for each interval is `max(0, Net Load) * Price`.

By summing these costs over each day of the year, we find the maximum daily cost. For the **`AddedEVScenario`**, this value is **€846.79**.

### 2.2. Action Space Diameter Calculation

The action space diameter is the Euclidean distance between the minimum and maximum possible actions across all controllable devices. In the `AddedEVScenario`, the agent controls the battery (`ESS1`).

- **Battery (`ESS1`) Action Range**: `[-1.5, 1.5]` kW

The diameter is calculated as:
$d = \sqrt{(1.5 - (-1.5))^2} = \sqrt{3^2} = 3.0$


### 2.3. The Reference Point

The final reference point (`ref_point`) is a vector combining the worst-case cost and the action space diameter. The training script uses these values to define the objective space for the MORL agent:

`ref_point = np.array([-846.79, -3.0])`

The full derivation of the Bounds and Diameters can be found in the notebooks `PCN-Bounds-AddedEVScenario.ipynb`, `PCN-Diameter-AddedEVScenario.ipynb`, `PCN-Bounds-ConstantPricesScenario.ipynb` and `PCN-Diameter-ConstantPricesScenario.ipynb` which are located in `./commonpower/notebooks`

---

## 3. Training the RL Agents

This is the main section of the notebook. Here, you can configure and run the training process for the different RL algorithms. 

You can change the following parameters to train different agents:
- `SCENARIO`: The power system environment to use. We'll use `AddedEVScenario`.
- `APPROACH`: The safeguarding mechanism. We'll use `WithProjectionSafeguard`.
- `PENALTY`: The penalty function for the safeguard. We'll use `DDPenalty`.
- `ALGORITHM`: The RL algorithm to train. You can choose between `PPO`, `PCN`, `GPIPD`, and `CAPQL`.
- `SEEDS`: A list of random seeds to use for training, allowing for robust evaluation.

Adjust the following to your liking. This will affect training and deployment. You can try out all the algorithms here.

In [ ]:
# --- Training Configuration ---
SCENARIO = Scenario.AddedEVScenario
APPROACH = Approach.WithProjectionSafeguard
PENALTY  = Penalty.DDPenalty
ALGORITHM = RLAlgorithm.CAPQL               # <- switch here: PPO / PCN / GPIPD / CAPQL
SEEDS = [1]                                 # use one seed for demonstration

# --- Hyperparameters ---
N_EPISODES       = 10                       # small demo
FORECAST_LENGTH  = 6                        # hours
START_TIME       = "2016-07-01 00:00:00"
END_TIME         = "2016-07-03 23:00:00"
DEVICE           = "cpu"
PREFERENCE_VECTOR = np.array([0.5, 0.5])    # only used when NOT MORL

Then we can run the training...

In [ ]:
# --- Derived Configuration ---
date_format     = "%Y-%m-%d %H:%M:%S"
start_dt        = datetime.strptime(START_TIME, date_format)
end_dt          = datetime.strptime(END_TIME,   date_format)
episode_length  = int(1 + (end_dt - start_dt).total_seconds() // 3600)
forecast_horizon = timedelta(hours=FORECAST_LENGTH)

# --- Forecaster ---
forecaster = PersistenceForecaster(
    frequency=timedelta(hours=1),
    horizon=forecast_horizon,
    look_back=timedelta(hours=24)
)

policy_cls = ALGORITHM.to_policy_class()

# --- Train per seed ---
run_id = f"{SCENARIO.name}/{APPROACH.name}/{PENALTY.name}/{ALGORITHM.name}"

for seed in SEEDS:
    train_sys = create_scenario_and_controller(
        stage=Stage.Train,
        scenario_constructor=SCENARIO.value,
        approach=APPROACH,
        penalty=PENALTY,
        forecast_length=FORECAST_LENGTH,
        forecaster=forecaster
    )

    # MORL vs single-obj
    is_morl = bool(getattr(policy_cls, "is_morl", False))
    ref_point = np.array([-846.79, -3.0]) if is_morl else None
    scalarisation_fn = (lambda r: np.dot(r, PREFERENCE_VECTOR)) if not is_morl else None

    # Algorithm-specific config (minimal & consistent)
    if ALGORITHM == RLAlgorithm.PPO:
        algo_config = PPO_Config(
            device=DEVICE, n_steps=episode_length, batch_size=episode_length,
            learning_rate=0.008, n_epochs=5
        )
    elif ALGORITHM == RLAlgorithm.PCN:
        algo_config = PCN_Config(device=DEVICE, batch_size=episode_length)
    elif ALGORITHM == RLAlgorithm.CAPQL:
        algo_config = CAPQL_Config(device=DEVICE, batch_size=episode_length, eval_freq=1e9, checkpoints=True)
    elif ALGORITHM == RLAlgorithm.GPIPD:
        algo_config = GPIPD_Config(device=DEVICE, batch_size=episode_length, eval_freq=1e9, checkpoints=True)
    else:
        raise NotImplementedError(ALGORITHM)

    run_training(
        run_id=run_id,
        algo_config=algo_config,
        forecast_horizon=forecast_horizon,
        episode_length=episode_length,
        train_sys=train_sys,
        rl_algorithm=ALGORITHM,
        seed=seed,
        n_episodes=N_EPISODES,
        start_time=START_TIME,
        scalarisation_fn=scalarisation_fn,
        ref_point=ref_point
    )

---

## 4. Deployment and Evaluation

After training, we deploy the learned policies and evaluate their performance. We will compare our trained agent against a baseline Model Predictive Controller (MPC). 

In [ ]:
# --- Deployment Configuration ---
EVAL_PERIOD  = "2016-01-02 00:00:00"   # ≥ 24h after 2016-01-01 due to 24h look-back
N_EVAL_STEPS = 7 * 24
EVAL_SEED    = 42

Then we can run our deployment...

In [ ]:
policy_cls = ALGORITHM.to_policy_class()

# Build a lightweight deployment config matching ALGORITHM
if ALGORITHM == RLAlgorithm.PPO:
    dep_algo_config = PPO_Config(device=DEVICE, n_steps=1, batch_size=1, learning_rate=0.008, n_epochs=5)
elif ALGORITHM == RLAlgorithm.PCN:
    dep_algo_config = PCN_Config(device=DEVICE, batch_size=1)
elif ALGORITHM == RLAlgorithm.CAPQL:
    dep_algo_config = CAPQL_Config(device=DEVICE, batch_size=1, eval_freq=1e9, checkpoints=True)
elif ALGORITHM == RLAlgorithm.GPIPD:
    dep_algo_config = GPIPD_Config(device=DEVICE, batch_size=1, eval_freq=1e9, checkpoints=True)
else:
    raise NotImplementedError(ALGORITHM)

# Recreate scenario (Deploy mode)
rl_save_path = f"{SCENARIO.name}/{APPROACH.name}/{PENALTY.name}/{ALGORITHM.name}"
rl_sys = create_scenario_and_controller(
    stage=Stage.Deploy,
    scenario_constructor=SCENARIO.value,
    approach=APPROACH,
    penalty=PENALTY,
    forecast_length=FORECAST_LENGTH,
    forecaster=forecaster
)

# Optional: PCN steering targets (safe no-op for other algos)
pcn_desired_return  = globals().get("pcn_desired_return", None)
pcn_desired_horizon = globals().get("pcn_desired_horizon", None)

print(f"\n--- DEPLOYING {ALGORITHM.name} ---")
# Try both signatures of run_deployment
try:
    rl_history, rl_log = run_deployment(
        scenario=rl_sys,
        train_seed=SEEDS[0],
        horizon=forecast_horizon,
        approach=APPROACH,
        rl_algorithm=ALGORITHM,
        algo_config=dep_algo_config,
        save_path=rl_save_path,
        eval_period=EVAL_PERIOD,
        n_eval_steps=N_EVAL_STEPS,
        eval_seed=EVAL_SEED,
    )
except TypeError:
    rl_history = run_deployment(
        scenario=rl_sys,
        train_seed=SEEDS[0],
        horizon=forecast_horizon,
        approach=APPROACH,
        rl_algorithm=ALGORITHM,
        algo_config=dep_algo_config,
        save_path=rl_save_path,
        eval_period=EVAL_PERIOD,
        n_eval_steps=N_EVAL_STEPS,
        eval_seed=EVAL_SEED,
    )
    rl_log = None  # will be loaded from CSV in the results cell

print("\n--- DEPLOYING Optimal Controller (Baseline) ---")
opt_save_path = f"{SCENARIO.name}/{Approach.OptimalController.name}"
opt_sys = create_scenario_and_controller(
    stage=Stage.Deploy,
    scenario_constructor=SCENARIO.value,
    approach=Approach.OptimalController,
    penalty=Penalty.NoPenalty,
    forecast_length=FORECAST_LENGTH,
    forecaster=forecaster
)
try:
    opt_history, opt_log = run_deployment(
        scenario=opt_sys,
        train_seed=1,
        horizon=forecast_horizon,
        approach=Approach.OptimalController,
        rl_algorithm=None,
        algo_config=None,
        save_path=opt_save_path,
        eval_period=EVAL_PERIOD,
        n_eval_steps=N_EVAL_STEPS,
        eval_seed=EVAL_SEED,
    )
except TypeError:
    opt_history = run_deployment(
        scenario=opt_sys,
        train_seed=1,
        horizon=forecast_horizon,
        approach=Approach.OptimalController,
        rl_algorithm=None,
        algo_config=None,
        save_path=opt_save_path,
        eval_period=EVAL_PERIOD,
        n_eval_steps=N_EVAL_STEPS,
        eval_seed=EVAL_SEED,
    )
    opt_log = None  # will be loaded from CSV in the results cell

Now that both the RL agent and the optimal controller have been deployed, we evaluate their **economic performance**.  
We do this by reading the deployment logs (`seed_results.csv`) that were generated for each controller.  

From these logs, we extract the **total cost** incurred during the evaluation horizon and compare:

- The cumulative cost of the trained RL agent.
- The cumulative cost of the optimal controller baseline.
- The difference between the two.

This allows us to quantify whether the RL controller achieved comparable or improved cost efficiency compared to the baseline.
That way, the flow is:

In [ ]:
# --- Deployment Cost Evaluation (results only) ---
from pathlib import Path

results_base = Path.cwd() / "results"

# Paths to the CSVs written by deployment
rl_csv  = results_base / f"{SCENARIO.name}/{APPROACH.name}/{PENALTY.name}/{ALGORITHM.name}" / str(SEEDS[0]) / "seed_results.csv"
opt_csv = results_base / f"{SCENARIO.name}/{Approach.OptimalController.name}" / "1" / "seed_results.csv"

# Use in-memory logs if the previous cell returned them; otherwise read CSVs
if "rl_log" not in globals() or rl_log is None:
    rl_log  = pd.read_csv(rl_csv)
if "opt_log" not in globals() or opt_log is None:
    opt_log = pd.read_csv(opt_csv)

def total_cost_from_log(df: pd.DataFrame) -> float:
    if "cum_cost" in df.columns:
        return float(df["cum_cost"].iloc[-1])
    if "cost" in df.columns:
        return float(df["cost"].sum())
    for alt in ("cum_total_cost", "total_cost"):
        if alt in df.columns:
            return float(df[alt].iloc[-1])
    raise KeyError(f"Could not find cost columns in log. Columns: {list(df.columns)[:20]}")

rl_cost_total  = total_cost_from_log(rl_log)
opt_cost_total = total_cost_from_log(opt_log)

print(f"Total cost (RL – {ALGORITHM.name}): {rl_cost_total:.2f} €")
print(f"Total cost (Optimal Controller):     {opt_cost_total:.2f} €")
print(f"Difference (RL − Optimal):           {rl_cost_total - opt_cost_total:+.2f} €")


## 5. Results Visualization

Finally, let's visualize the performance of the deployed controllers. We plot:

- **Cumulative cost difference to the baseline** (top): how much the RL agent saves or spends vs. the Optimal Controller over time.
- **Safety performance** (bottom): normalized cumulative interventions (fallback to cumulative penalty if interventions aren’t logged).  
  A lower curve means fewer safety interventions; curves are normalized to [0–1] for comparability.

In [ ]:
def plot_safety_vs_cost(rl_log, opt_log, rl_name):
    
    # --- timestamps (match what actually got logged) ---
    start_dt = datetime.strptime(EVAL_PERIOD, "%Y-%m-%d %H:%M:%S")
    steps = min(len(rl_log), len(opt_log), N_EVAL_STEPS)
    ts = pd.to_datetime([start_dt + timedelta(hours=i) for i in range(steps)])

    # --- helpers to read/derive columns safely ---
    def get_series(df, names):
        for n in names:
            if n in df.columns: 
                return pd.Series(df[n]).iloc[:steps]
        return None

    def step_cost_from(df):
        s = get_series(df, ["cost"])
        if s is not None:
            return s
        s_cum = get_series(df, ["cum_cost"])
        if s_cum is not None:
            s = s_cum.diff()
            s.iloc[0] = s_cum.iloc[0]
            return s
        raise KeyError(f"Log missing cost columns. Has: {list(df.columns)[:15]}...")

    # --- COST: RL <-> baseline ---
    rl_step_cost  = step_cost_from(rl_log)
    opt_step_cost = step_cost_from(opt_log)
    rl_cum  = rl_step_cost.cumsum()
    opt_cum = opt_step_cost.cumsum()
    cost_diff = (rl_cum - opt_cum)

    # --- SAFETY: normalized cumulative interventions ---
    rl_interv  = get_series(rl_log,  ["cum_interventions"])
    opt_interv = get_series(opt_log, ["cum_interventions"])

    def normalize(v):
        v = pd.Series(v).iloc[:steps]
        final = float(v.iloc[-1]) if len(v) else 0.0
        return v / final if final > 0 else v*0.0

    rl_safety  = normalize(rl_interv)
    opt_safety = normalize(opt_interv)

    # --- plot ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12), sharex=True)

    # Top: cost difference
    ax1.plot(ts, cost_diff.values, label=f"{rl_name} − Baseline")
    ax1.axhline(0.0, linestyle="--", color="gray", label="Baseline")
    ax1.set_ylabel("Cost Difference to Baseline (€)")
    ax1.set_title("Relative Economic Performance")
    ax1.grid(True, linestyle="--", linewidth=0.5)
    ax1.legend()

    # Bottom: safety performance (normalized)
    ax2.plot(ts, rl_safety.values, label=f"{rl_name}")
    ax2.plot(ts, opt_safety.values, label="Optimal Controller", linestyle="--")
    ax2.set_ylabel("Normalized Cumulative Interventions")
    ax2.set_title("Safety Performance")
    ax2.grid(True, linestyle="--", linewidth=0.5)
    ax2.set_ylim(bottom=0)
    ax2.legend()

    plt.xlabel("Time")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

plot_safety_vs_cost(rl_log, opt_log, ALGORITHM.name)

---

# Troubleshooting

### GurobiDirect import error (Pyomo) 

This is just a quick “ugly” workaround. If you see an error like:

```
pyomo.common.errors.ApplicationError: No Python bindings available for GurobiDirect solver plugin
```

that’s not our code it’s a bug in Pyomo’s `GurobiDirect` plugin import check. Under some conditions it incorrectly decides that `gurobipy` is unavailable even when it is installed and working.

#### One-cell fix (recommended for running this notebook)

Run this **before** you create or solve any Pyomo models:

In [ ]:
# Ugly but effective workaround for Pyomo+Gurobi import check
import gurobipy
from pyomo.solvers.plugins.solvers import gurobi_direct as gd

# Force the plugin to recognize gurobipy
gd.gurobipy = gurobipy
gd.gurobipy_available = True

print("Patched Pyomo GurobiDirect -> gurobipy_available =", gd.gurobipy_available)

This bypasses the fragile import wrapper and lets Pyomo use your Gurobi installation normally.

#### Alternative (manual) fix for local debugging only

Edit the file in your virtual environment:

* Windows: `.venv\Lib\site-packages\pyomo\solvers\plugins\solvers\gurobi_direct.py`
* macOS/Linux: `.venv/lib/pythonX.Y/site-packages/pyomo/solvers/plugins/solvers/gurobi_direct.py`

Replace the block that uses `attempt_import('gurobipy', ...)` with:

```python
    import gurobipy  # force a normal import
    gurobipy_available = True  # hard-override the availability flag
```


#### Clean up / revert

To undo the manual edit: re-install Pyomo in your venv:

  ```
  pip install --force-reinstall "pyomo==<your-version>"
  ```

Again: this is an upstream library issue with the Pyomo plugin’s import logic, **not** a bug in our project.
